In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_01_contacts_all
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_01_contacts_all
# Spec Ref  : §1.2 Refresh Materialized Views
# Purpose   : Build contacts_all Delta table = contacts UNION ALL crm_contacts
#             (currently just hgi.silver.contacts — ready for multi-source union)
#             This MUST run FIRST before any other map notebook.
#             All downstream map queries join against contacts_all.
#
# Why Delta table instead of plain VIEW?
#   A plain VIEW re-scans hgi.silver.contacts on every downstream JOIN.
#   A Delta table is read once, cached on disk, and skipped by Liquid Clustering.
#   For 100k+ contacts this is the difference between seconds and minutes.
#
# Serverless: works on 2X-Small — safe_spark_conf skips unsupported configs
# Job Compute: all SPARK_CONF_MAP configs apply for full performance
# =============================================================================

In [0]:

# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2 ── Widget
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
tenant_id   = tenant_id_from_customer(customer_id)

print(f"=== Map 01: contacts_all ===  customer={customer_id}  tenant={tenant_id}")
print(f"  Reading from : {sv}.contacts")
print(f"  Writing to   : {sv}.contacts_all")

In [0]:
# ✅ H1a FIX: Add CDF version tracking to avoid full table scans
# Create version tracking metadata table (once)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {sv}.map_metadata (
        key STRING,
        version BIGINT,
        last_updated TIMESTAMP
    ) USING DELTA
""")

# Get last processed version
try:
    last_ver = spark.sql(f"""
        SELECT version FROM {sv}.map_metadata
        WHERE key = 'contacts_all_last_contacts_version'
    """).collect()[0][0]
    print(f"📍 Last processed contacts version: {last_ver}")
except:
    last_ver = 0
    print(f"📍 First run - will process all data")

# Get current version
current_ver = spark.sql(f"""
    SELECT MAX(version) FROM (DESCRIBE HISTORY {sv}.contacts)
""").collect()[0][0]

print(f"📍 Current contacts version: {current_ver}")

# Skip if no changes
if current_ver == last_ver:
    print(f"✅ No new changes in contacts table - skipping map_01")
    pm.save(status="SUCCESS", rows_processed=0)
    dbutils.notebook.exit("0")

print(f"🔄 Processing versions {last_ver} → {current_ver}")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"

import sys, os
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
if os.path.abspath(project_root) not in sys.path:
    sys.path.append(os.path.abspath(project_root))

from utils.pipeline_metrics import PipelineMetrics
pm = PipelineMetrics(
    spark          = spark,
    parent_run_id  = parent_run_id,
    job_name       = "02b_map_01_contacts_all",
    task_key       = "run_job_D_silver_map",
    source_system  = source_system,
    load_type      = load_type,
    customer_id    = customer_id,
    object_name    = object_name,
)
pm.init()

initialize_audit_tables()

In [0]:
# CELL 3 ── Incremental MERGE for contacts_all (tenant-scoped)
# ✅ H1a FIX: Use CDF to read only changed records instead of full table
try:
    # Safe drop in case target exists as a VIEW
    safe_drop_view(f"{sv}.contacts_all")

    # Ensure table exists (idempotent)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {sv}.contacts_all (
            id                    STRING NOT NULL,
            tenant                BIGINT,
            email                 STRING,
            domain                STRING,
            source_system         STRING,
            source_system_object  STRING,
            source_key_name       STRING,
            source_key_value      STRING,
            data_timestamp        TIMESTAMP,
            a_accountid           STRING
        )
        USING DELTA
        CLUSTER BY (tenant, email, domain)
        {DELTA_TBLPROPS_MAP}
    """)

    # ✅ H1a FIX: Read CDF instead of full table on incremental runs
    if last_ver > 0:
        print(f"📚 Reading CDF from {sv}.contacts (versions {last_ver} → {current_ver})")
        source_df = spark.read.format("delta") \
            .option("readChangeFeed", "true") \
            .option("startingVersion", last_ver + 1) \
            .option("endingVersion", current_ver) \
            .table(f"{sv}.contacts") \
            .filter("_change_type IN ('insert', 'update_postimage')") \
            .filter(f"tenant = {tenant_id}")
        
        # Register as temp view for MERGE
        source_df.createOrReplaceTempView("contacts_source")
        source_table = "contacts_source"
        cdf_rows = source_df.count()
        print(f"✅ CDF records to process: {cdf_rows:,}")
    else:
        print(f"📚 First run - reading full {sv}.contacts table")
        source_table = f"{sv}.contacts"

    # Incremental MERGE: update changed, insert new, delete removed
    merge_query = f"""
        MERGE INTO {sv}.contacts_all AS target
        USING (
            SELECT id, tenant, email, domain, source_system, source_system_object,
                   source_key_name, source_key_value, data_timestamp, a_accountid
            FROM {source_table}
            WHERE id IS NOT NULL AND tenant = {tenant_id}
        ) AS source
        ON target.id = source.id AND target.tenant = source.tenant
        WHEN MATCHED AND source.data_timestamp > target.data_timestamp THEN UPDATE SET
            target.email = source.email,
            target.domain = source.domain,
            target.source_system = source.source_system,
            target.source_system_object = source.source_system_object,
            target.source_key_name = source.source_key_name,
            target.source_key_value = source.source_key_value,
            target.data_timestamp = source.data_timestamp,
            target.a_accountid = source.a_accountid
        WHEN NOT MATCHED THEN INSERT (id, tenant, email, domain, source_system,
            source_system_object, source_key_name, source_key_value, data_timestamp, a_accountid)
            VALUES (source.id, source.tenant, source.email, source.domain, source.source_system,
                source.source_system_object, source.source_key_name, source.source_key_value,
                source.data_timestamp, source.a_accountid)
        WHEN NOT MATCHED BY SOURCE AND target.tenant = {tenant_id} THEN DELETE
    """
    
    spark.sql(merge_query)
    print(f"✅ MERGE completed successfully")
    
except Exception as e:
    print(f"\u274c contacts_all build failed: {e}")
    log_audit(
        job_name       = "02b_map_01_contacts_all",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
# ✅ H1a FIX: Update version tracking after successful MERGE
try:
    spark.sql(f"""
        MERGE INTO {sv}.map_metadata t
        USING (
            SELECT 'contacts_all_last_contacts_version' AS key,
                   {current_ver} AS version,
                   current_timestamp() AS last_updated
        ) s
        ON t.key = s.key
        WHEN MATCHED THEN UPDATE SET 
            t.version = s.version, 
            t.last_updated = s.last_updated
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✅ Version tracking updated: {current_ver}")
except Exception as e:
    print(f"⚠️ Version tracking update failed (non-critical): {e}")

In [0]:
# CELL 4 ── Verify and report
try:
    n = cnt(f"{sv}.contacts_all")
    print(f"  contacts_all: {n:,} rows built")

    # Spec DQ gate: no null IDs
    null_ids = spark.sql(f"SELECT COUNT(*) FROM {sv}.contacts_all WHERE id IS NULL").collect()[0][0]
    if null_ids > 0:
        print(f"  WARNING: {null_ids} null IDs in contacts_all")
    else:
        print(f"  DQ OK: no null IDs")
except Exception as e:
    print(f"❌ DQ verification failed: {e}")
    log_audit(
        job_name       = "02b_map_01_contacts_all",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "DQ_CHECK_FAILED",
        error_type     = type(e).__name__,
        error_reason   = f"DQ check failed: {str(e)[:450]}",
    )
    raise

In [0]:
try:
    total_rows = spark.table(f"{sv}.contacts_all").count()
    pm.save(status="SUCCESS", rows_processed=total_rows)
except Exception as e:
    print(f"❌ Metrics save failed: {e}")
    log_audit(
        job_name       = "02b_map_01_contacts_all",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise